# Beauty360 — Modèle : détecter les problèmes de peau

**À lancer dans Google Colab** (*Exécution > Modifier le type d'exécution > GPU T4*). Durée : environ 25 minutes.

**Le but :** à partir d'une photo du visage, dire quels problèmes on voit : acné, points noirs, points blancs, taches brunes, pores dilatés, rougeurs, peau grasse, peau sèche, rides, poches.

**Les données (vraies photos) :** dataset « skincare » sur Roboflow Universe (version 7), licence **CC BY 4.0** :
https://universe.roboflow.com/skincare-gxqjb/skincare-8dmzr — 2 350 photos, déjà découpées en train / valid / test.

**Type de problème :** classification **multi-label** = une photo peut avoir PLUSIEURS problèmes à la fois.
Donc une sortie **sigmoïde** par problème (probabilité indépendante), et non un softmax (un seul choix).

**Démarche :** on entraîne **2 modèles** (EfficientNetB0 et MobileNetV2), on les **compare sur VALID**, on garde le meilleur, puis on l'évalue **une seule fois sur TEST**.

## 1. Télécharger les données
Remplace `TA_CLE_API` par ta clé (Roboflow > Settings > API Keys). Ne la publie jamais sur GitHub.

In [ ]:
!pip install -q roboflow iterative-stratification
from roboflow import Roboflow

rf = Roboflow(api_key="TA_CLE_API")
projet = rf.workspace("skincare-gxqjb").project("skincare-8dmzr")
dataset = projet.version(7).download("folder")   # un dossier par étiquette (seul format proposé)
DOSSIER = dataset.location
print(DOSSIER)

## 2. Lire les étiquettes et explorer
Roboflow range ce dataset en **un dossier par combinaison d'étiquettes** (ex. `Acne Blackheads`). On découpe le nom du dossier pour retrouver toutes les étiquettes d'une photo, puis on crée une colonne par problème (1 = présent, 0 = absent).

**Amélioration 1 : un nouveau découpage stratifié.** Dans le découpage d'origine de Roboflow, TEST ne contient **aucune** photo de rougeurs ni de points blancs : impossible de les évaluer. On réunit tout, puis on redécoupe en 70 / 15 / 15 en gardant **la même proportion de chaque problème** dans train, valid et test (bibliothèque `iterative-stratification`, faite pour le multi-label).

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Les noms des dossiers donnent les étiquettes :
# "Acne Blackheads" -> la photo a de l'acné ET des points noirs (multi-label)
print("Dossiers trouvés :", sorted(os.listdir(os.path.join(DOSSIER, "train"))))

def lire(partie):
    photos = {}
    racine = os.path.join(DOSSIER, partie)
    for nom_dossier in sorted(os.listdir(racine)):
        dossier = os.path.join(racine, nom_dossier)
        if not os.path.isdir(dossier):
            continue
        etiquettes = nom_dossier.split()          # on coupe aux espaces
        for f in os.listdir(dossier):
            # si une même photo est dans plusieurs dossiers, on additionne ses étiquettes
            photo = photos.setdefault(f, {"filename": f, "chemin": os.path.join(dossier, f), "etiquettes": set()})
            photo["etiquettes"].update(etiquettes)
    return pd.DataFrame(photos.values())

# On réunit les 3 dossiers : le découpage d'origine laisse 0 photo de rougeurs
# et 0 photo de points blancs dans TEST, donc impossible de les évaluer.
tout = pd.concat([lire("train"), lire("valid"), lire("test")], ignore_index=True)
tout = tout.drop_duplicates("filename").reset_index(drop=True)   # une photo = une seule fois
CLASSES = sorted(set().union(*tout["etiquettes"]))

# Une colonne par problème : 1 = présent, 0 = absent
for c in CLASSES:
    tout[c] = tout["etiquettes"].apply(lambda e: int(c in e))

# Nouveau découpage STRATIFIÉ multi-label : 70 % train / 15 % valid / 15 % test,
# avec la même proportion de chaque problème dans les 3 parties.
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
coupe1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
i_train, i_reste = next(coupe1.split(tout, tout[CLASSES].values))
reste = tout.iloc[i_reste].reset_index(drop=True)
coupe2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
i_valid, i_test = next(coupe2.split(reste, reste[CLASSES].values))
train = tout.iloc[i_train].reset_index(drop=True)
valid, test = reste.iloc[i_valid].reset_index(drop=True), reste.iloc[i_test].reset_index(drop=True)

print("Classes :", CLASSES)
print("Photos :", len(train), "train /", len(valid), "valid /", len(test), "test")

# Photos par problème : une classe rare sera moins bien apprise
compte = pd.DataFrame({p: d[CLASSES].sum() for p, d in [("train", train), ("valid", valid), ("test", test)]})
print(compte)
compte["train"].sort_values().plot.barh(title="Photos par problème (train)"); plt.show()
print("Nombre moyen de problèmes par photo :", round(train[CLASSES].sum(axis=1).mean(), 2))

In [ ]:
# Quelques exemples avec leurs étiquettes (regarde aussi la diversité des teintes de peau)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, ligne) in zip(axes.flat, train.sample(8, random_state=0).iterrows()):
    ax.imshow(plt.imread(ligne["chemin"]))
    ax.set_title(", ".join(c for c in CLASSES if ligne[c] == 1), fontsize=8)
    ax.axis("off")
plt.show()

## 3. Préparer les images
- Taille **224×224**, valeurs **0 à 255** (chaque modèle fait sa propre normalisation à l'intérieur)
- **Augmentation** sur train seulement : retournement, rotation, zoom, luminosité, contraste (photos prises avec des téléphones et lumières différents)
- **Amélioration 2 : sur-échantillonnage.** Les problèmes rares (taches brunes, rougeurs…) sont recopiés dans train (jusqu'à 4 copies) pour que le modèle les voie plus souvent. Valid et test ne sont **jamais** modifiés : on évalue sur la vraie répartition.

In [ ]:
import tensorflow as tf
TAILLE, LOT = 224, 32
tf.keras.utils.set_random_seed(42)       # pour que les résultats soient reproductibles

def charger(chemin, etiquettes):
    image = tf.io.decode_image(tf.io.read_file(chemin), channels=3, expand_animations=False)
    image = tf.image.resize(image, (TAILLE, TAILLE))
    return tf.cast(image, tf.float32), etiquettes

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.15),
])

def creer_dataset(df, melanger=False):
    ds = tf.data.Dataset.from_tensor_slices((df["chemin"].values, df[CLASSES].values.astype("float32")))
    if melanger:
        ds = ds.shuffle(len(df), seed=42)
    ds = ds.map(charger, num_parallel_calls=tf.data.AUTOTUNE).batch(LOT)
    if melanger:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

# Amélioration 2 : SUR-ÉCHANTILLONNAGE des problèmes rares (seulement dans train !)
# Une photo d'un problème rare est montrée plusieurs fois au modèle ;
# grâce à l'augmentation, chaque copie est un peu différente (rotation, lumière...).
def surechantillonner(df, cible=300, max_copies=4):
    morceaux = [df]
    for c in CLASSES:
        n = df[c].sum()
        if 0 < n < cible:
            copies = min(max_copies, int(np.ceil(cible / n)) - 1)
            morceaux += [df[df[c] == 1]] * copies
    return pd.concat(morceaux, ignore_index=True)

train_equilibre = surechantillonner(train)
print(pd.DataFrame({"avant": train[CLASSES].sum(), "après": train_equilibre[CLASSES].sum()}))

ds_train = creer_dataset(train_equilibre, True)
ds_valid, ds_test = creer_dataset(valid), creer_dataset(test)
y_valid, y_test = valid[CLASSES].values, test[CLASSES].values

## 4. Une fonction pour construire et entraîner un modèle
**Transfer learning** : on part d'un réseau déjà entraîné sur ImageNet (il sait reconnaître formes et textures) et on ajoute une tête avec **une sortie sigmoïde par problème**.

Entraînement en 2 étapes :
1. base **gelée**, on n'apprend que la tête (taux d'apprentissage 1e-3) ;
2. **fine-tuning** : on dégèle les 40 dernières couches et on réentraîne doucement (1e-5).

*EarlyStopping* arrête quand l'AUC de validation ne progresse plus, et garde les meilleurs poids.

In [ ]:
def construire(nom):
    entree = tf.keras.Input((TAILLE, TAILLE, 3))
    if nom == "EfficientNetB0":
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                    input_shape=(TAILLE, TAILLE, 3))
        x = entree                                        # EfficientNet normalise lui-même (0-255)
    else:  # MobileNetV2 attend des valeurs entre -1 et 1
        base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                                 input_shape=(TAILLE, TAILLE, 3))
        x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entree)
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    sortie = tf.keras.layers.Dense(len(CLASSES), activation="sigmoid")(x)
    return tf.keras.Model(entree, sortie), base


def entrainer(nom, epoques=15):
    modele, base = construire(nom)
    metriques = [tf.keras.metrics.AUC(multi_label=True, name="auc")]
    arret = tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4,
                                             restore_best_weights=True)
    debut = time.time()

    # Étape 1 : la tête seule
    modele.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metriques)
    h1 = modele.fit(ds_train, validation_data=ds_valid, epochs=epoques, callbacks=[arret], verbose=2)

    # Étape 2 : fine-tuning des 40 dernières couches (BatchNorm gelées = plus stable)
    base.trainable = True
    for couche in base.layers[:-40]:
        couche.trainable = False
    for couche in base.layers:
        if isinstance(couche, tf.keras.layers.BatchNormalization):
            couche.trainable = False
    modele.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=metriques)
    h2 = modele.fit(ds_train, validation_data=ds_valid, epochs=epoques, callbacks=[arret], verbose=2)

    historique = {"auc": h1.history["auc"] + h2.history["auc"],
                  "val_auc": h1.history["val_auc"] + h2.history["val_auc"],
                  "coupure": len(h1.history["auc"])}
    return modele, historique, round(time.time() - debut)

## 5. Entraîner les 2 modèles

In [ ]:
modeles, historiques, durees = {}, {}, {}
for nom in ["EfficientNetB0", "MobileNetV2"]:
    print("=" * 20, nom, "=" * 20)
    modeles[nom], historiques[nom], durees[nom] = entrainer(nom)

# Courbes d'apprentissage côte à côte
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, nom in zip(axes, modeles):
    h = historiques[nom]
    ax.plot(h["auc"], label="train"); ax.plot(h["val_auc"], label="valid")
    ax.axvline(h["coupure"] - 0.5, ls="--", c="gray")   # début du fine-tuning
    ax.set_title(f"{nom} : AUC par époque"); ax.legend()
plt.show()

## 6. Choisir un seuil par problème, puis comparer sur VALID
Le modèle donne une probabilité. À partir de quel % dit-on « oui, il y a de l'acné » ?
Pour chaque problème, on cherche le seuil qui donne le meilleur **F1** sur VALID.

On compare ensuite les 2 modèles **sur VALID uniquement** : le TEST reste caché jusqu'à la fin, sinon l'évaluation serait truquée.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

def choisir_seuils(proba, y):
    seuils = {}
    for i, c in enumerate(CLASSES):
        candidats = np.arange(0.2, 0.81, 0.05)
        scores = [f1_score(y[:, i], proba[:, i] >= s, zero_division=0) for s in candidats]
        seuils[c] = round(float(candidats[int(np.argmax(scores))]), 2)
    return seuils

def appliquer(proba, seuils):
    return np.column_stack([proba[:, i] >= seuils[c] for i, c in enumerate(CLASSES)]).astype(int)

def taille_tflite(modele):
    conv = tf.lite.TFLiteConverter.from_keras_model(modele)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    return conv.convert()

comparaison, seuils_par_modele, tflites = [], {}, {}
for nom, modele in modeles.items():
    proba = modele.predict(ds_valid, verbose=0)
    seuils = choisir_seuils(proba, y_valid)
    pred = appliquer(proba, seuils)
    seuils_par_modele[nom] = seuils
    tflites[nom] = taille_tflite(modele)

    # Temps de réponse pour UNE photo (ce que vivra l'utilisatrice)
    une_photo = next(iter(ds_valid))[0][:1]
    modele.predict(une_photo, verbose=0)                      # 1er appel = chauffe
    t = time.time(); [modele.predict(une_photo, verbose=0) for _ in range(10)]
    comparaison.append({
        "modele": nom,
        "f1_macro_valid": round(f1_score(y_valid, pred, average="macro", zero_division=0), 3),
        "f1_micro_valid": round(f1_score(y_valid, pred, average="micro", zero_division=0), 3),
        "auc_macro_valid": round(roc_auc_score(y_valid, proba, average="macro"), 3),
        "taille_tflite_mo": round(len(tflites[nom]) / 1e6, 1),
        "ms_par_photo": round((time.time() - t) / 10 * 1000),
        "duree_entrainement_s": durees[nom],
    })

comparaison = pd.DataFrame(comparaison).set_index("modele")
comparaison

In [ ]:
# On garde le modèle avec le meilleur F1 macro sur VALID
# (macro = moyenne des F1 de chaque problème : les problèmes rares comptent autant que les fréquents)
MEILLEUR = comparaison["f1_macro_valid"].idxmax()
modele, SEUILS = modeles[MEILLEUR], seuils_par_modele[MEILLEUR]
print("Modèle retenu :", MEILLEUR)
print("Seuils :", SEUILS)
comparaison[["f1_macro_valid", "auc_macro_valid"]].plot.bar(rot=0, title="Comparaison sur VALID", ylim=(0, 1))
plt.show()

## 7. Évaluation HONNÊTE sur le TEST (une seule fois)
Les 244 photos de test n'ont jamais été vues. Ce sont ces chiffres-là qu'on présente au jury.
- **Précision** : quand le modèle dit « acné », a-t-il raison ?
- **Rappel** : sur toutes les photos avec acné, combien en trouve-t-il ?
- **F1** : l'équilibre des deux. **AUC** : capacité à séparer « présent » / « absent », quel que soit le seuil.

In [ ]:
from sklearn.metrics import classification_report
proba_test = modele.predict(ds_test, verbose=0)
pred_test = appliquer(proba_test, SEUILS)

rapport = classification_report(y_test, pred_test, target_names=CLASSES, zero_division=0, output_dict=True)
print(classification_report(y_test, pred_test, target_names=CLASSES, zero_division=0))
auc_test = {c: (round(float(roc_auc_score(y_test[:, i], proba_test[:, i])), 3)
                if y_test[:, i].min() != y_test[:, i].max() else None)
            for i, c in enumerate(CLASSES)}
print("AUC par problème :", auc_test)
print("Peu de photos de test (<10), résultat peu fiable :",
      [c for i, c in enumerate(CLASSES) if y_test[:, i].sum() < 10])

pd.DataFrame({c: [rapport[c]["precision"], rapport[c]["recall"], rapport[c]["f1-score"]] for c in CLASSES},
             index=["précision", "rappel", "F1"]).T.plot.barh(figsize=(8, 6), title=f"{MEILLEUR} sur TEST")
plt.show()

## 8. Exporter en TFLite (léger, pour Render) et vérifier
Quantification « dynamic range » : fichier ~4 fois plus petit, réponse quasi identique. On vérifie sur 20 vraies photos de test.

In [ ]:
tflite = tflites[MEILLEUR]
open("skin_problems.tflite", "wb").write(tflite)
print(f"skin_problems.tflite : {len(tflite) / 1e6:.1f} Mo")

interp = tf.lite.Interpreter(model_content=tflite); interp.allocate_tensors()
e, s = interp.get_input_details()[0]["index"], interp.get_output_details()[0]["index"]
ecarts = []
for chemin in test["chemin"].values[:20]:
    img = charger(chemin, 0)[0][None].numpy()
    interp.set_tensor(e, img); interp.invoke()
    ecarts.append(np.abs(interp.get_tensor(s)[0] - modele.predict(img, verbose=0)[0]).max())
print("Écart maximum Keras / TFLite :", round(float(max(ecarts)), 4))

## 9. Enregistrer et télécharger
L'API lit `skin_problems.json` (ordre des classes, seuils, résultats). Mets les 2 fichiers dans `C:\\beauty\\beauty360\\models\\`.

In [ ]:
infos = {
    "dataset": "Roboflow Universe - skincare (skincare-gxqjb/skincare-8dmzr) v7, CC BY 4.0",
    "modele": MEILLEUR,
    "taille_image": TAILLE,
    "classes": CLASSES,
    "seuils": SEUILS,
    "comparaison_valid": comparaison.reset_index().to_dict(orient="records"),
    "test": {
        "nb_photos": int(len(test)),
        "f1_micro": round(rapport["micro avg"]["f1-score"], 3),
        "f1_macro": round(rapport["macro avg"]["f1-score"], 3),
        "par_classe": {c: {"precision": round(rapport[c]["precision"], 3), "rappel": round(rapport[c]["recall"], 3),
                           "f1": round(rapport[c]["f1-score"], 3), "nb_photos": int(rapport[c]["support"]),
                           "auc": auc_test[c]} for c in CLASSES},
    },
}
json.dump(infos, open("skin_problems.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps({k: infos[k] for k in ("modele", "comparaison_valid", "test")}, indent=2))

from google.colab import files
files.download("skin_problems.tflite")
files.download("skin_problems.json")